# Data labelling

In this notebook, you will label the data - in other words, you will tell AI what is in the pictures, so that it can learn to recognise them!

## About Jupyter Notebook

This is a Jupyter Notebook. It contains different *cells*.

The cells have different types. For example, this text is written in a Markdown cell.

Below is a code cell:

In [ ]:
print("Hello from Jupyter Notebook!")

You can run the cell by putting your cursor there and doing one of the following:
1. Clicking the run button at the top of the screen
2. Pressing ctrl-enter
3. Pressing shift-enter, which will execute the current cell and advance to the next one

To complete the workshop, go through each cell, try to understand what it does, and execute it!

## Imports

First we import necessary Python libraries

In [ ]:
import ipywidgets as widgets
from pathlib import Path

# File handling

Labeling is often done in specialized applications. In our case we will use a simple Python program instead. First we need to tell the program where it should look for the data.

In [ ]:
raw_data_dir = Path("raw_data")  # Our raw data directory
forward_data_dir = Path("labeled_data/forward")  # The photos with clear forward path will go here
obstacle_data_dir = Path("labeled_data/obstacle")  # The photos with obstacles will go here

In [ ]:
# Create the directories if they don't exist
forward_data_dir.mkdir(exist_ok=True)
obstacle_data_dir.mkdir(exist_ok=True)

# Function to get the next photo from the raw data directory
def get_next_photo():
  try:
    return next(raw_data_dir.iterdir()).name
  except StopIteration:
    return None

curr_photo = get_next_photo()

# Count how many photos we have in total and how many are labeled
total_photo_count = len(list(raw_data_dir.iterdir())) + len(list(forward_data_dir.iterdir())) + len(list(obstacle_data_dir.iterdir()))
photos_labeled = total_photo_count - len(list(raw_data_dir.iterdir()))

print(f"Created directories: {forward_data_dir}, {obstacle_data_dir}")
print(f"Total photos: {total_photo_count}")
print(f"Photos labeled: {photos_labeled}")

## Labeling UI

This long cell creates the UI for our labeling process. Try to run it first and see what happens!

In [ ]:
# Set up the widgets
# Image widget to display the current photo
image_widget = widgets.Image(
  value=b"",
  format="jpg",
  width=600,
  height=600,
)

# Header to show the current photo count
header = widgets.HTML(
  value=f"",
  layout=widgets.Layout(margin='10px 0px 20px 0px')
)

# Buttons for labeling photos with a clear forward path
forward_button = widgets.Button(
  description="Road clear, go forward!",
  layout=widgets.Layout(width='300px', height='100px'),
  button_style="success",
  style={'font_weight': 'bold', 'font_size': '20px'}
)

# Button for labeling photos with obstacles
obstacle_button = widgets.Button(
  description="Obstacle detected!",
  layout=widgets.Layout(width='300px',height='100px'),
  button_style="danger",
  style={'font_weight': 'bold', 'font_size': '20px'}
)

# Arrange the widgets
hbox = widgets.HBox([forward_button, obstacle_button])
vbox = widgets.VBox([header, image_widget, hbox])

# When we click a button, move the current photo to the appropriate directory and update the display
def update_image():
  global curr_photo
  curr_photo = get_next_photo()

  global photos_labeled
  photos_labeled = total_photo_count - len(list(raw_data_dir.iterdir()))

  # Update the header and image widget
  if curr_photo:
    header.value = f"<h2>Photo {photos_labeled + 1}/{total_photo_count}</h2>"
    image_widget.value = (raw_data_dir / curr_photo).read_bytes()
  else:
    header.value = "<h2>All photos labeled!</h2>"
    vbox.children = [header]
    image_widget.value = b""

# Button click handlers
def on_forward_button_clicked(_b):
  (raw_data_dir / curr_photo).rename(forward_data_dir / curr_photo)
  print("Moved to forward_data")
  update_image()

def on_obstacle_button_clicked(_b):
  (raw_data_dir / curr_photo).rename(obstacle_data_dir / curr_photo)
  print("Moved to obstacle_data")
  update_image()

forward_button.on_click(on_forward_button_clicked)
obstacle_button.on_click(on_obstacle_button_clicked)

update_image()

display(vbox)

Above you can see a photo and two buttons. If you think this photo represents a clear road and the robot should move forward, press the green button. If you think the road is blocked and the robot should turn, press the red button. Think well, as this is one of the most important parts of training our AI model!

When you label all the images, move to the next notebook - `2_model_training.ipynb`